In [1]:
# sofar i've run the sleep_lab table and hypoxia operations in three parts but had a bug in in, only selecting :8000 for hypoxia calculation. 
# therefore:
# read in 3 sleeplab_tables, concat.
# read in 3 hypoxia tables, concat. (less rows because of bug)
# add hypoxia values to sleeplab tables.
# for all files where hypoxia value is missing, run hypoxia computation.

In [2]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
import datetime
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

sys.path.append('/home/wolfgang/repos/sleep_general')
from mgh_sleeplab import load_mgh_signal, annotations_preprocess, vectorize_respiratory_events, vectorize_sleep_stages

In [3]:
st0 = pd.read_csv('sleeplab_table_0.csv')
st1 = pd.read_csv('sleeplab_table_1.csv')
st2 = pd.read_csv('sleeplab_table_2.csv')

assert st0.shape[1] == st1.shape[1]
assert st0.shape[1] == st2.shape[1]

st = pd.concat([st0, st1, st2], axis=0)
st.drop(['Unnamed: 0'], axis=1, inplace=True)
st.reset_index(drop=True, inplace=True)

In [4]:
print(st.shape)
st.head(3)

(25231, 12)


,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations
0,Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...
2,Morris_Martha_112512_2118.000,Z9992290,512-71-58,Morris,Martha,Female,1947-04-30,2012-11-25,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...


In [5]:
ht0 = pd.read_csv('sleeplab_table_hypoxia_0.csv')
ht1 = pd.read_csv('sleeplab_table_hypoxia_1.csv')
ht2 = pd.read_csv('sleeplab_table_hypoxia_2.csv')

assert ht0.shape[1] == ht1.shape[1]
assert ht0.shape[1] == ht2.shape[1]

ht = pd.concat([ht0, ht1, ht2], axis=0)
ht.drop(['Unnamed: 0'], axis=1, inplace=True)
ht.reset_index(drop=True, inplace=True)

In [6]:
print(ht.shape)
ht.head(3)

(23831, 18)


,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
0,Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.641667,0.3,0.7,0.000000,0.000000,0.000000
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333
2,Morris_Martha_112512_2118.000,Z9992290,512-71-58,Morris,Martha,Female,1947-04-30,2012-11-25,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,0.883333,0.0,0.0,5.814500,4.109917,1.704583


In [7]:
hypoxia_columns = list(ht.columns[-6:])
print(hypoxia_columns)

['Total_Sleep_Time', 'AHI', 'HypoxiaBurden', 'T90_minutes', 'T90_minutes_desat', 'T90_minutes_unspecific']


In [8]:
st.set_index('FolderName', inplace=True)
ht.set_index('FolderName', inplace=True)
sleeplab_table = st.join(ht[hypoxia_columns])

# example check
# st.loc["Penta_Christina_091109_2149.000"]
# ht.loc["Penta_Christina_091109_2149.000"]
# table.loc["Penta_Christina_091109_2149.000"]

In [9]:
sleeplab_table.head(5)

,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
FolderName,,,,,,,,,,,,,,,,,
Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.641667,0.3,0.7,0.000000,0.000000,0.000000
Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333
Morris_Martha_112512_2118.000,Z9992290,512-71-58,Morris,Martha,Female,1947-04-30,2012-11-25,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,0.883333,0.0,0.0,5.814500,4.109917,1.704583
Schlein_Toby_012717_2300.000,Z6488130,354-22-12,Schlein,Toby,Female,1945-11-16,2017-01-27,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.350000,8.4,6.4,6.542250,6.542250,0.000000
Callahan_Marian_042010_2158.000,Z10588347,160-39-24,Callahan,Marian,Female,1959-08-30,2010-04-20,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.366667,2.8,2.1,0.815583,0.199500,0.616083


In [10]:
from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes

def compute_hypoxia_statistics(data, apnea, sleep_stage, fs):
    
    data['apnea'] = apnea

    dt_start = pd.Timestamp('2000-01-01 00:00:00')
    dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
    pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
    data.index = pseudo_dt_index

    data = compute_spo2_clean(data, fs=fs)
    data['spo2'] = data['spo2_clean']

    data['apnea_binary'] = np.isin(data['apnea'],[1,2,3,4]).astype(int)
    data['apnea_end'] = np.isin(data['apnea_binary'].diff(), [-1])
    
    sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
    hours_sleep = sum(sleep_stage<5)/fs/3600
    
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep, apnea_name = 'apnea')
    
    T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)
    
    AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)

    return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [11]:
sleeplab_table.shape

(25231, 17)

In [12]:
part = 4

for jloc, row in tqdm(sleeplab_table.iterrows()):
    
    try:
        if pd.isna(row.path_signal) | pd.isna(row.path_annotations):
            continue
        
        if pd.notna(row.AHI):
            continue # already done.
        
        if 'natus' in row.Path.lower():
            continue # natus still running and not affected by bug.
        
        # read in raw data:
        signal, params = load_mgh_signal(row.path_signal)
        annotations = pd.read_csv(row.path_annotations)

        fs = int(params['Fs'])
        signal_len = signal.shape[0]

        # get respiratory event and sleep stage array:
        annotations = annotations_preprocess(annotations, fs)
        resp = vectorize_respiratory_events(annotations, signal_len)
        sleep_stage = vectorize_sleep_stages(annotations, signal_len)

        hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)

        sleeplab_table.loc[jloc, 'AHI'] = ahi
        sleeplab_table.loc[jloc, 'HypoxiaBurden'] = hypoxia_burden
        sleeplab_table.loc[jloc, 'Total_Sleep_Time'] = total_sleep_time
        sleeplab_table.loc[jloc, 'T90_minutes'] = T90burden
        sleeplab_table.loc[jloc, 'T90_minutes_desat'] = T90desaturation
        sleeplab_table.loc[jloc, 'T90_minutes_unspecific'] = T90nonspecific

        sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')
        
    except Exception as e:
        print(f'{jloc}, {row.Path}')
        print(e)

sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')

0it [00:00, ?it/s]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
82it [00:40,  1.84it/s]

Hexley_Allie_121015_0834.000, M:\Datasets_ConvertedData\sleeplab\grass_data\Hexley_Allie_121015_0834.000
local variable 'samples_limit' referenced before assignment


135it [00:40,  4.09it/s]

Rummel_Jeremy_010811_0553.000, M:\Datasets_ConvertedData\sleeplab\grass_data\Rummel_Jeremy_010811_0553.000
Cannot convert non-finite values (NA or inf) to integer
Dugan_David_040314_2202.000, M:\Datasets_ConvertedData\sleeplab\grass_data\Dugan_David_040314_2202.000
Cannot convert non-finite values (NA or inf) to integer


142it [00:51,  2.72it/s]

...

21219it [12:47:22,  4.57it/s]

Darrell_Kevin_050312_0118.000, M:\Datasets_ConvertedData\sleeplab\grass_data\Darrell_Kevin_050312_0118.000
Cannot convert non-finite values (NA or inf) to integer


24436it [12:47:45, 111.99it/s]

Kolchinsky_Evelina_090210_0819.000, M:\Datasets_ConvertedData\sleeplab\grass_data\Kolchinsky_Evelina_090210_0819.000
index 5930000 is out of bounds for axis 0 with size 5930000


25231it [12:47:45,  1.83s/it] 


In [ ]:
sleeplab_table = sleeplab_table.reset_index()

In [16]:
sleeplab_table.shape

(25231, 17)

In [17]:
st.shape

(25231, 11)

In [18]:
# last step: add the natus files that have been computed in parallel to the above table.

In [27]:
ht2_new = pd.read_csv('sleeplab_table_hypoxia_2.csv')
ht2_new.drop(['Unnamed: 0'], axis=1, inplace=True)

In [28]:
ht2_new.shape

(7831, 18)

In [32]:
sleeplab_table.set_index('FolderName', inplace=True)
ht2_new.set_index('FolderName', inplace=True)

In [39]:
sleeplab_table.loc[ht2_new.index, hypoxia_columns] = ht2_new[hypoxia_columns].values

In [41]:
sleeplab_table.to_csv('sleeplab_table_hypoxia.csv')

In [42]:
sleeplab_table.columns

Index(['PatientID', 'MRN', 'LastName', 'FirstName', 'Sex', 'DateOfBirth',
       'DateOfVisit', 'TypeOfTest', 'Path', 'path_signal', 'path_annotations',
       'Total_Sleep_Time', 'AHI', 'HypoxiaBurden', 'T90_minutes',
       'T90_minutes_desat', 'T90_minutes_unspecific'],
      dtype='object')

### Found a big in hypoxic bruden code that resulted in HB==0. so therefore, run the code again for all patients with HB==0.

In [44]:
sleeplab_table.reset_index(inplace=True)

In [45]:
sleeplab_table.head(2)

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
0,Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.641667,0.3,0.7,0.000000,0.000000,0.000000
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333


In [46]:
part = 5
n_patients_fixed = 0

for jloc, row in tqdm(sleeplab_table.iterrows()):
    
    try:
        if pd.isna(row.path_signal) | pd.isna(row.path_annotations):
            continue
        
        if row.HypoxiaBurden != 0:
            continue
                    
        # read in raw data:
        signal, params = load_mgh_signal(row.path_signal)
        annotations = pd.read_csv(row.path_annotations)

        fs = int(params['Fs'])
        signal_len = signal.shape[0]

        # get respiratory event and sleep stage array:
        annotations = annotations_preprocess(annotations, fs)
        resp = vectorize_respiratory_events(annotations, signal_len)
        sleep_stage = vectorize_sleep_stages(annotations, signal_len)

        hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)

        n_patients_fixed += 1

        sleeplab_table.loc[jloc, 'AHI'] = ahi
        sleeplab_table.loc[jloc, 'HypoxiaBurden'] = hypoxia_burden
        sleeplab_table.loc[jloc, 'Total_Sleep_Time'] = total_sleep_time
        sleeplab_table.loc[jloc, 'T90_minutes'] = T90burden
        sleeplab_table.loc[jloc, 'T90_minutes_desat'] = T90desaturation
        sleeplab_table.loc[jloc, 'T90_minutes_unspecific'] = T90nonspecific

        sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')
        
    except Exception as e:
        print(f'{jloc}, {row.Path}')
        print(e)

sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')

67it [04:44,  3.88s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
310it [23:22,  5.48s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
673it [41:24,  3.68s/it]

672, M:\Datasets_ConvertedData\sleeplab\grass_data\Webster_Ann_063009_2211.000
axis 1 is out of bounds for array of dimension 1


717it [44:03,  3.54s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
758it [47:48,  4.65s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
857it [56:12,  5.30s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
885it [58:46,  4.52s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
1081it [1:17:18,  5.74s/it]

1080, M:\Datasets_ConvertedData\sleeplab\grass_data\Connelly_Carolyn_F._062915_2314.000
axis 1 is out of bounds for array of dimension 1


1135it [1:21:00,  4.59s/it]

...

18060it [19:21:27,  4.69s/it]/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:437: RuntimeWarning: Mean of empty slice
  mean_spo2_firsthalf = mean_spo2[:mean_spo2.shape[0]//2]
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:233: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:194: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
18090it [19:23:23,  3.75s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
18587it [19:55:47,  2.75s/it]

18586, M:\Datasets_ConvertedData\sleeplab\grass_data\Parnes_Joseph_110909_2145.000
axis 1 is out of bounds for array of dimension 1


18800it [20:09:29,  3.34s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
18974it [20:19:27,  2.40s/it]

18973, M:\Datasets_ConvertedData\sleeplab\grass_data\Denuccio_Alyssa_031818_2137.000
axis 1 is out of bounds for array of dimension 1


19019it [20:22:36,  3.81s/it]/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:233: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:194: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
19558it [20:55:40,  4.78s/it]<ipython-input-10-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
19761it [21:13:47, 11.41s/it]/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:437: RuntimeWarning: Mean of empty slice
  mean_spo2_firsthalf = mean_spo2[:mean_spo2.shape[0]//2]
/home/wolfgang/anacon

In [10]:
# fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# ax[0].plot(resp)
# ax[0].plot(sleep_stage)
# ax[1].plot(signal['spo2'])
# ax[1].set_ylim([80,100])
# # ax[2].plot(signal['ptaf'])
# ax[3].plot(signal['chest'])
# ax[3].plot(signal['abd'])